# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [1]:
import os
import json
import pickle
import time

import numpy as np
import optuna
import pandas as pd
import trackio
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [2]:
# Ensure required output folders exist
os.makedirs("models", exist_ok=True)
os.makedirs("plots", exist_ok=True)

train_df = pd.read_csv("Dataset/train.csv")
test_df = pd.read_csv("Dataset/test.csv")

# TODO: Implement your preprocessing here (use LabelEncoder or manual dictionaries)
# Ensure you keep all necessary features that will be shown on the UI dashboard.
target_col = "price"
feature_cols = [col for col in train_df.columns if col != target_col]

# Data-quality guardrails for fair evaluation.
initial_train_rows = int(len(train_df))
initial_test_rows = int(len(test_df))

train_dup_count = int(train_df.duplicated().sum())
test_dup_count = int(test_df.duplicated().sum())
train_df = train_df.drop_duplicates().reset_index(drop=True)
test_df = test_df.drop_duplicates().reset_index(drop=True)

# Remove train rows that share identical feature vectors with test rows.
# This avoids optimistic metrics from feature-level leakage across train/test files.
train_feature_keys = train_df[feature_cols].astype(str).agg("||".join, axis=1)
test_feature_keys = set(test_df[feature_cols].astype(str).agg("||".join, axis=1).tolist())
overlap_mask = train_feature_keys.isin(test_feature_keys)
feature_overlap_removed = int(overlap_mask.sum())
train_df = train_df.loc[~overlap_mask].reset_index(drop=True)

raw_train_df = train_df.copy()
raw_test_df = test_df.copy()

# Identify categorical columns from training data
categorical_cols = train_df[feature_cols].select_dtypes(include=["object"]).columns.tolist()

label_encoders = {}
label_mappings = {}

for col in categorical_cols:
    le = LabelEncoder()
    train_values = train_df[col].astype(str)
    le.fit(train_values)

    mapping = {cls: int(idx) for idx, cls in enumerate(le.classes_)}

    train_df[col] = train_values.map(mapping).astype(int)
    # Unknown or missing categories in test are mapped to -1
    test_df[col] = test_df[col].astype(str).map(mapping).fillna(-1).astype(int)

    label_encoders[col] = le
    label_mappings[col] = mapping

# Fill missing numeric values with train medians for consistent preprocessing
numeric_cols = [col for col in feature_cols if col not in categorical_cols]
for col in numeric_cols:
    median_val = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)

unseen_category_report = {}
for col in categorical_cols:
    train_set = set(raw_train_df[col].astype(str))
    test_set = set(raw_test_df[col].astype(str))
    unseen_values = sorted(test_set - train_set)
    unseen_category_report[col] = {
        "count": int(len(unseen_values)),
        "examples": unseen_values[:10],
        "handling": "unseen_or_missing_mapped_to_-1_in_test_preprocessing",
    }

data_quality_report = {
    "initial_rows": {"train": initial_train_rows, "test": initial_test_rows},
    "duplicate_rows_removed": {"train": train_dup_count, "test": test_dup_count},
    "feature_overlap_rows_removed_from_train": feature_overlap_removed,
    "final_rows": {"train": int(len(train_df)), "test": int(len(test_df))},
}

if "BHK" in train_df.columns:
    bhk_values = sorted(set(train_df["BHK"].dropna().astype(int).tolist()))
    data_quality_report["bhk_unique_values_train"] = bhk_values
    data_quality_report["bhk_binary_flag"] = bool(set(bhk_values).issubset({0, 1}))

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Feature columns:", feature_cols)
print("Categorical columns encoded:", categorical_cols)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Data quality summary:")
print(
    f"- Duplicates removed -> train: {train_dup_count}, test: {test_dup_count}; "
    f"feature-overlap removed from train: {feature_overlap_removed}"
)
print("Unseen categorical values in test (mapped to -1):")
for col in categorical_cols:
    info = unseen_category_report[col]
    print(f"- {col}: {info['count']} unseen, examples={info['examples'][:3]}")

# Save encoder and report artifacts for Streamlit + auditability
with open("models/label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

with open("models/label_mappings.json", "w", encoding="utf-8") as f:
    json.dump(label_mappings, f, indent=2)

with open("models/unseen_category_report.json", "w", encoding="utf-8") as f:
    json.dump(unseen_category_report, f, indent=2)

with open("models/data_quality_report.json", "w", encoding="utf-8") as f:
    json.dump(data_quality_report, f, indent=2)

Feature columns: ['location', 'city', 'latitude', 'longitude', 'numBathrooms', 'numBalconies', 'isNegotiable', 'SecurityDeposit', 'Status', 'Size_ft²', 'BHK', 'rooms_num', 'property_type', 'verification_days']
Categorical columns encoded: ['location', 'city', 'Status', 'property_type']
Train shape: (9089, 14) Test shape: (2630, 14)
Data quality summary:
- Duplicates removed -> train: 1439, test: 152; feature-overlap removed from train: 600
Unseen categorical values in test (mapped to -1):
- location: 57 unseen, examples=['AGCR Enclave', 'Antarli', 'B1 Block Paschim Vihar']
- city: 0 unseen, examples=[]
- Status: 0 unseen, examples=[]
- property_type: 0 unseen, examples=[]


## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

In [3]:
rf = RandomForestRegressor(random_state=42)

# TODO: Initialize trackio project/experiment here
trackio_session = None
trackio_ready = False


def to_native_types(obj):
    if isinstance(obj, dict):
        return {k: to_native_types(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_native_types(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def trackio_init():
    global trackio_session, trackio_ready

    project_name = "STT-A4-Rent-Prediction"
    run_name = "task1-hpo"

    # Try common Trackio init/start signatures across versions.
    signature_candidates = [
        {"project": project_name, "experiment": run_name},
        {"project": project_name, "run": run_name},
        {"project": project_name, "name": run_name},
        {"project": project_name},
        {},
    ]

    callables = []
    if hasattr(trackio, "init") and callable(trackio.init):
        callables.append(trackio.init)
    if hasattr(trackio, "start") and callable(trackio.start):
        callables.append(trackio.start)

    if not callables:
        print("Trackio unavailable: no init/start function found")
        trackio_ready = False
        return

    last_error = None
    for init_fn in callables:
        for kwargs in signature_candidates:
            try:
                trackio_session = init_fn(**kwargs)
                trackio_ready = True
                print(f"Trackio initialized using {init_fn.__name__} with args: {kwargs}")
                return
            except TypeError as e:
                # Signature mismatch; keep trying other combinations.
                last_error = e
                continue
            except Exception as e:
                last_error = e
                break

    trackio_ready = False
    if last_error is not None:
        print(f"Trackio init warning: {last_error}")


def trackio_log(payload):
    if not trackio_ready:
        return

    native_payload = to_native_types(payload)

    try:
        if hasattr(trackio, "log") and callable(trackio.log):
            trackio.log(native_payload)
            return
        if trackio_session is not None and hasattr(trackio_session, "log") and callable(trackio_session.log):
            trackio_session.log(native_payload)
            return
        print("Trackio log warning: no compatible log function found")
    except Exception as e:
        print(f"Trackio log warning: {e}")


trackio_init()

grid_param_grid = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [10, 15, 20, 25, 30],
    "min_samples_split": [2, 5, 8],
}

random_param_dist = {
    "n_estimators": np.arange(50, 201),
    "max_depth": np.arange(10, 31),
    "min_samples_split": np.arange(2, 11),
}

results = {}
trial_logs = []

# TODO: 1. Grid Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score
start = time.time()
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=grid_param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)
grid_time = time.time() - start

grid_errors = -grid_search.cv_results_["mean_test_score"]
for i, params in enumerate(grid_search.cv_results_["params"]):
    entry = {
        "method": "grid_search",
        "trial": int(i + 1),
        "mae": float(grid_errors[i]),
        "best_so_far_mae": float(np.min(grid_errors[: i + 1])),
        "params": to_native_types(params),
    }
    trial_logs.append(entry)
    trackio_log(entry)
grid_best_curve = np.minimum.accumulate(grid_errors)

results["grid"] = {
    "best_params": grid_search.best_params_,
    "best_cv_mae": -grid_search.best_score_,
    "time_sec": grid_time,
    "iterations": len(grid_search.cv_results_["params"]),
    "curve": grid_best_curve.tolist(),
}

trackio_log({
    "method": "grid_search",
    "time_sec": grid_time,
    "iterations": results["grid"]["iterations"],
    "best_cv_mae": results["grid"]["best_cv_mae"],
    "best_params": results["grid"]["best_params"],
})

# TODO: 2. Random Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score
start = time.time()
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=random_param_dist,
    n_iter=60,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
random_search.fit(X_train, y_train)
random_time = time.time() - start

random_errors = -random_search.cv_results_["mean_test_score"]
for i, params in enumerate(random_search.cv_results_["params"]):
    entry = {
        "method": "random_search",
        "trial": int(i + 1),
        "mae": float(random_errors[i]),
        "best_so_far_mae": float(np.min(random_errors[: i + 1])),
        "params": to_native_types(params),
    }
    trial_logs.append(entry)
    trackio_log(entry)
random_best_curve = np.minimum.accumulate(random_errors)

results["random"] = {
    "best_params": random_search.best_params_,
    "best_cv_mae": -random_search.best_score_,
    "time_sec": random_time,
    "iterations": len(random_search.cv_results_["params"]),
    "curve": random_best_curve.tolist(),
}

trackio_log({
    "method": "random_search",
    "time_sec": random_time,
    "iterations": results["random"]["iterations"],
    "best_cv_mae": results["random"]["best_cv_mae"],
    "best_params": results["random"]["best_params"],
})

# TODO: 3. Bayesian Optimization (Optuna) Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score
optuna_history = []


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 10, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "random_state": 42,
        "n_jobs": -1,
    }
    model = RandomForestRegressor(**params)
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=1,
    )
    mae = -scores.mean()
    optuna_history.append(mae)
    entry = {
        "method": "bayesian_optuna",
        "trial": int(len(optuna_history)),
        "mae": float(mae),
        "best_so_far_mae": float(np.min(optuna_history)),
        "params": to_native_types(params),
    }
    trial_logs.append(entry)
    trackio_log(entry)
    return mae


start = time.time()
study = optuna.create_study(
    direction="minimize",
    study_name="rf_bayesian_optimization",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=60, show_progress_bar=False)
optuna_time = time.time() - start

optuna_best_curve = np.minimum.accumulate(np.array(optuna_history))

results["bayesian"] = {
    "best_params": study.best_params,
    "best_cv_mae": study.best_value,
    "time_sec": optuna_time,
    "iterations": len(study.trials),
    "curve": optuna_best_curve.tolist(),
}

trackio_log({
    "method": "bayesian_optuna",
    "time_sec": optuna_time,
    "iterations": results["bayesian"]["iterations"],
    "best_cv_mae": results["bayesian"]["best_cv_mae"],
    "best_params": results["bayesian"]["best_params"],
})

print("Optimization complete.")
for method_name, info in results.items():
    print(method_name, "->", info["best_cv_mae"], info["best_params"])

* Trackio project initialized: STT-A4-Rent-Prediction
* Trackio metrics logged to: C:\Users\Saharsh\.cache\huggingface\trackio
* Created new run: task1-hpo


Trackio initialized using init with args: {'project': 'STT-A4-Rent-Prediction', 'name': 'task1-hpo'}
Fitting 5 folds for each of 60 candidates, totalling 300 fits
Fitting 5 folds for each of 60 candidates, totalling 300 fits


[I 2026-04-15 00:56:31,673] A new study created in memory with name: rf_bayesian_optimization
[I 2026-04-15 00:56:35,381] Trial 0 finished with value: 15639.418222699547 and parameters: {'n_estimators': 106, 'max_depth': 29, 'min_samples_split': 8}. Best is trial 0 with value: 15639.418222699547.
[I 2026-04-15 00:56:39,488] Trial 1 finished with value: 15797.771689861256 and parameters: {'n_estimators': 140, 'max_depth': 13, 'min_samples_split': 3}. Best is trial 0 with value: 15639.418222699547.
[I 2026-04-15 00:56:41,646] Trial 2 finished with value: 15671.549236027393 and parameters: {'n_estimators': 58, 'max_depth': 28, 'min_samples_split': 7}. Best is trial 0 with value: 15639.418222699547.
[I 2026-04-15 00:56:45,237] Trial 3 finished with value: 16758.23545675743 and parameters: {'n_estimators': 156, 'max_depth': 10, 'min_samples_split': 10}. Best is trial 0 with value: 15639.418222699547.
[I 2026-04-15 00:56:50,560] Trial 4 finished with value: 15683.032327752964 and parameters:

Optimization complete.
grid -> 15496.296940769094 {'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 150}
random -> 15474.491311468704 {'n_estimators': np.int64(183), 'min_samples_split': np.int64(3), 'max_depth': np.int64(30)}
bayesian -> 15470.252896078022 {'n_estimators': 177, 'max_depth': 28, 'min_samples_split': 2}


In [4]:
# Optional: log a compact summary table so Media & Tables has a clear artifact for screenshots.
comparison_rows = []
for method_name, info in results.items():
    comparison_rows.append(
        {
            "method": method_name,
            "best_cv_mae": float(info["best_cv_mae"]),
            "iterations": int(info["iterations"]),
            "time_sec": float(info["time_sec"]),
            "best_params": json.dumps(to_native_types(info["best_params"]), sort_keys=True),
        }
    )

comparison_df = pd.DataFrame(comparison_rows).sort_values("best_cv_mae").reset_index(drop=True)
print("\nHPO summary table:")
print(comparison_df)

if trackio_ready and hasattr(trackio, "Table"):
    try:
        trackio_log({"hpo_summary_table": trackio.Table(dataframe=comparison_df)})
        trial_df = pd.DataFrame(
            [
                {
                    "method": t["method"],
                    "trial": t["trial"],
                    "mae": t["mae"],
                    "best_so_far_mae": t["best_so_far_mae"],
                    "params": json.dumps(t["params"], sort_keys=True),
                }
                for t in trial_logs
            ]
        )
        trackio_log({"hpo_trial_table": trackio.Table(dataframe=trial_df)})
        print("Logged hpo_summary_table to Trackio Media & Tables.")
    except Exception as e:
        print(f"Trackio table log warning: {e}")
else:
    print("Trackio table logging skipped: Trackio not initialized or Table API unavailable.")


HPO summary table:
     method   best_cv_mae  iterations    time_sec  \
0  bayesian  15470.252896          60  365.367294   
1    random  15474.491311          60  230.407018   
2      grid  15496.296941          60  253.207841   

                                         best_params  
0  {"max_depth": 28, "min_samples_split": 2, "n_e...  
1  {"max_depth": 30, "min_samples_split": 3, "n_e...  
2  {"max_depth": 30, "min_samples_split": 2, "n_e...  
Logged hpo_summary_table to Trackio Media & Tables.


## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [5]:
# TODO: Generate and save trials_vs_error.png
# X-axis: Number of iterations
# Y-axis: Best CV error found so far
# Overlay Grid, Random, and Bayesian methods on the same plot.

plt.figure(figsize=(10, 6))

for method_key, label in [("grid", "Grid Search"), ("random", "Random Search"), ("bayesian", "Bayesian (Optuna)")]:
    curve = results[method_key]["curve"]
    x_vals = np.arange(1, len(curve) + 1)
    plt.plot(x_vals, curve, marker="o", linewidth=2, markersize=3, label=label)

plt.title("Trials vs Best CV MAE")
plt.xlabel("Number of Iterations/Trials")
plt.ylabel("Best Mean CV Error (MAE)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("plots/trials_vs_error.png", dpi=200)
plt.show()


# TODO: Generate and save optuna_hyperparameter_space.png
# Build a clear 2x3 panel containing only off-diagonal parameter pairs.
try:
    import optuna.visualization as ov
    from plotly.subplots import make_subplots

    pair_order = [
        ("n_estimators", "max_depth"),
        ("n_estimators", "min_samples_split"),
        ("max_depth", "n_estimators"),
        ("max_depth", "min_samples_split"),
        ("min_samples_split", "n_estimators"),
        ("min_samples_split", "max_depth"),
    ]

    subplot_titles = [f"{x} vs {y}" for x, y in pair_order]
    contour_grid = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.08,
        vertical_spacing=0.18,
    )

    for idx, (x_param, y_param) in enumerate(pair_order):
        row = (idx // 3) + 1
        col = (idx % 3) + 1
        pair_fig = ov.plot_contour(study, params=[x_param, y_param])

        for trace in pair_fig.data:
            contour_grid.add_trace(trace, row=row, col=col)

        contour_grid.update_xaxes(title_text=x_param, row=row, col=col)
        contour_grid.update_yaxes(title_text=y_param, row=row, col=col)

    # Show a single colorbar for readability.
    colorbar_kept = False
    for trace in contour_grid.data:
        if hasattr(trace, "showscale"):
            if colorbar_kept:
                trace.showscale = False
            else:
                trace.showscale = True
                colorbar_kept = True

    contour_grid.update_layout(
        title="Optuna Hyperparameter Space (Off-diagonal Contour Panels)",
        template="plotly_white",
        width=1800,
        height=1100,
        font=dict(size=14),
        margin=dict(l=80, r=80, t=110, b=80),
    )
    contour_grid.write_image("plots/optuna_hyperparameter_space.png", scale=2)
    print("Saved clear 6-panel Optuna contour plot to plots/optuna_hyperparameter_space.png")
except Exception as e:
    print(f"Optuna contour export warning: {e}")
    print("Falling back to optimization history (also allowed by spec).")
    try:
        hist_fig = ov.plot_optimization_history(study)
        hist_fig.update_layout(
            title="Optuna Optimization History",
            template="plotly_white",
            width=1200,
            height=700,
            font=dict(size=16),
            margin=dict(l=80, r=80, t=100, b=80),
        )
        hist_fig.write_image("plots/optuna_hyperparameter_space.png", scale=2)
        print("Saved Optuna optimization history to plots/optuna_hyperparameter_space.png")
    except Exception as e2:
        print(f"Optuna history export warning: {e2}")
        print("Could not generate Optuna visualization PNG. Install/verify plotly and kaleido.")

print("Saved plots:")
print("- plots/trials_vs_error.png")
print("- plots/optuna_hyperparameter_space.png")

C:\Users\Saharsh\AppData\Local\Temp\ipykernel_25856\3651573711.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved clear 6-panel Optuna contour plot to plots/optuna_hyperparameter_space.png
Saved plots:
- plots/trials_vs_error.png
- plots/optuna_hyperparameter_space.png


## 4. Final Testing & Model Saving
Report the best hyperparameters found, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [6]:
# TODO: Print the best hyperparameters found by all 3 methods
print("Best configurations:")
for method_name in ["grid", "random", "bayesian"]:
    print(f"{method_name}: params={results[method_name]['best_params']}, CV_MAE={results[method_name]['best_cv_mae']:.4f}")


# TODO: Train the best model found on the full X_train
best_method = min(results.items(), key=lambda kv: kv[1]["best_cv_mae"])[0]
best_params = results[best_method]["best_params"]
print(f"\nOverall best method: {best_method}")
print(f"Using params: {best_params}")

best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)


# TODO: Evaluate the model on X_test (Report MAE)
test_preds = best_model.predict(X_test)
test_mae = mean_absolute_error(y_test, test_preds)
print(f"Final Test MAE: {test_mae:.4f}")


# TODO: Save best_model.pkl and any necessary encoders to the models/ folder
with open("models/best_rf_model.pkl", "wb") as f:
    pickle.dump(best_model, f)


def to_native_types(obj):
    if isinstance(obj, dict):
        return {k: to_native_types(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_native_types(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    return obj


metadata = {
    "best_method": str(best_method),
    "best_params": to_native_types(best_params),
    "test_mae": float(test_mae),
    "feature_cols": feature_cols,
    "categorical_cols": categorical_cols,
    "data_quality": to_native_types(data_quality_report),
    "evaluation_note": "Train duplicates, test duplicates, and train rows with feature overlap against test were removed before tuning and final evaluation.",
    "results": {
        k: {
            "best_params": to_native_types(v["best_params"]),
            "best_cv_mae": float(v["best_cv_mae"]),
            "time_sec": float(v["time_sec"]),
            "iterations": int(v["iterations"]),
        }
        for k, v in results.items()
    },
}

with open("models/training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved model artifacts:")
print("- models/best_rf_model.pkl")
print("- models/label_encoders.pkl")
print("- models/label_mappings.json")
print("- models/unseen_category_report.json")
print("- models/data_quality_report.json")
print("- models/training_metadata.json")

Best configurations:
grid: params={'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 150}, CV_MAE=15496.2969
random: params={'n_estimators': np.int64(183), 'min_samples_split': np.int64(3), 'max_depth': np.int64(30)}, CV_MAE=15474.4913
bayesian: params={'n_estimators': 177, 'max_depth': 28, 'min_samples_split': 2}, CV_MAE=15470.2529

Overall best method: bayesian
Using params: {'n_estimators': 177, 'max_depth': 28, 'min_samples_split': 2}
Final Test MAE: 15057.3009

Saved model artifacts:
- models/best_rf_model.pkl
- models/label_encoders.pkl
- models/label_mappings.json
- models/unseen_category_report.json
- models/data_quality_report.json
- models/training_metadata.json
